# MIRROR — Encode Discharge Notes with ClinicalBERT (Mean Pooling)

**Run on Kaggle P100 GPU** (~2h for MIMIC-III, ~8h for MIMIC-IV)

**Method**: Splits each note into 512-token chunks with 128-token overlap,
encodes each chunk with Bio_ClinicalBERT using **mean pooling over non-pad tokens**
(not CLS token), then aggregates chunks using **norm-weighted averaging**
(not naive mean — avoids dilution for long notes with 10+ chunks).

Also computes `note_global_mean.npy` from training set for mean-centering (fixes ClinicalBERT anisotropy).

**Upload to Kaggle:**
- `data/processed/cohort_mimic3.pkl`
- `data/processed/notes_text_mimic3.pkl`
- `data/processed/records_final.pkl` (needed for train-set global mean)

**Download after completion:**
- `note_embeddings_mimic3.pkl` (N x 768 embeddings + has_note flags)
- `note_global_mean.npy` (768d training-set mean for centering)

In [ ]:
# Install dependencies
!pip install -q transformers torch

In [ ]:

import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
import os

# === CONFIG ===
MIMIC_VERSION = 3           # Change to 4 for MIMIC-IV
OUTPUT_DIR = Path("/kaggle/working")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
MAX_LENGTH = 512
OVERLAP = 128
MIN_CHUNK_TOKENS = 32

# === Auto-detect INPUT_DIR ===
# Kaggle mounts datasets at /kaggle/input/<dataset-slug>/
# The zip structure adds extra nesting — find cohort_mimic3.pkl wherever it lives.
INPUT_DIR = None
search_root = Path("/kaggle/input")
for dirpath, dirnames, filenames in os.walk(search_root):
    if "cohort_mimic3.pkl" in filenames:
        INPUT_DIR = Path(dirpath)
        break

if INPUT_DIR is None:
    # Print tree for debugging
    print("ERROR: cohort_mimic3.pkl not found anywhere under /kaggle/input/")
    print("Directory tree:")
    for dirpath, dirnames, filenames in os.walk(search_root):
        depth = str(dirpath).count(os.sep) - str(search_root).count(os.sep)
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/")
        for f in filenames[:5]:
            print(f"{indent}  {f}")
        if len(filenames) > 5:
            print(f"{indent}  ... and {len(filenames)-5} more")
    raise FileNotFoundError("cohort_mimic3.pkl not found under /kaggle/input/")

print(f"Device: {DEVICE}")
print(f"MIMIC version: {MIMIC_VERSION}")
print(f"Input dir (auto-detected): {INPUT_DIR}")


In [ ]:

# Load cohort and notes text
suffix = f"_mimic{MIMIC_VERSION}"
cohort_path = INPUT_DIR / f"cohort_mimic{MIMIC_VERSION}.pkl"
notes_path = INPUT_DIR / f"notes_text{suffix}.pkl"

if not cohort_path.exists():
    raise FileNotFoundError(f"Not found: {cohort_path}")

print(f"Loading cohort from {cohort_path}")
with open(cohort_path, "rb") as f:
    cohort = pickle.load(f)

hadm_ids = np.array(cohort["hadm_ids"])
print(f"Total admissions: {len(hadm_ids):,}")

print(f"Loading notes from {notes_path}")
with open(notes_path, "rb") as f:
    notes_data = pickle.load(f)

note_texts = notes_data["notes"]  # dict: hadm_id → text
print(f"Notes available: {len(note_texts):,}")


In [ ]:
# Load ClinicalBERT
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print("Model loaded.")

In [ ]:
def chunk_text(text, tokenizer, max_length=512, overlap=128, min_tokens=32):
    """Split text into overlapping token chunks."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    stride = max_length - overlap
    chunks = []
    for start in range(0, len(tokens), stride):
        chunk = tokens[start : start + max_length]
        if len(chunk) < min_tokens:
            break
        chunks.append(chunk)
    if not chunks and tokens:
        chunks.append(tokens[:max_length])
    return chunks


def encode_chunks(chunks, model, tokenizer, device, batch_size=16):
    """Encode token chunks with mean pooling, then norm-weighted aggregation.
    
    Mean pooling (over non-pad tokens per chunk) is better than CLS for
    patient-specific discriminative content. Norm-weighted aggregation
    across chunks avoids dilution for long notes (10+ chunks).
    """
    all_embeds = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        max_len = max(len(c) for c in batch)
        input_ids = []
        attention_masks = []
        for c in batch:
            padded = c + [tokenizer.pad_token_id] * (max_len - len(c))
            mask = [1] * len(c) + [0] * (max_len - len(c))
            input_ids.append(padded)
            attention_masks.append(mask)

        input_ids_t = torch.tensor(input_ids, dtype=torch.long, device=device)
        attention_mask_t = torch.tensor(attention_masks, dtype=torch.long, device=device)

        with torch.no_grad():
            output = model(input_ids=input_ids_t, attention_mask=attention_mask_t)
            # Mean pooling over non-pad tokens (better than CLS for patient-specific info)
            hidden = output.last_hidden_state  # (B, seq_len, 768)
            mask_expanded = attention_mask_t.unsqueeze(-1).float()  # (B, seq_len, 1)
            sum_hidden = (hidden * mask_expanded).sum(dim=1)  # (B, 768)
            count = mask_expanded.sum(dim=1).clamp(min=1)  # (B, 1)
            mean_embeds = (sum_hidden / count).cpu().numpy()  # (B, 768)
            all_embeds.append(mean_embeds)

    all_embeds = np.concatenate(all_embeds, axis=0)  # (num_chunks, 768)
    
    # Norm-weighted chunk aggregation: chunks with more informative content
    # tend to have higher-norm embeddings. Avoids dilution for long notes.
    if all_embeds.shape[0] == 1:
        return all_embeds[0]
    norms = np.linalg.norm(all_embeds, axis=1)  # (num_chunks,)
    weights = norms / (norms.sum() + 1e-8)
    return (weights[:, None] * all_embeds).sum(axis=0)  # (768,)


In [ ]:
# Encode all notes
N = len(hadm_ids)
embeddings = np.zeros((N, 768), dtype=np.float32)
has_note = np.zeros(N, dtype=np.float32)

hadm_to_idx = {int(h): i for i, h in enumerate(hadm_ids)}

processed = 0
skipped = 0

import time
t0 = time.time()

for hadm_id, text in note_texts.items():
    hadm_id = int(hadm_id)
    if hadm_id not in hadm_to_idx:
        skipped += 1
        continue
    if not text or len(text.strip()) < 50:
        skipped += 1
        continue

    idx = hadm_to_idx[hadm_id]
    chunks = chunk_text(text, tokenizer, MAX_LENGTH, OVERLAP, MIN_CHUNK_TOKENS)
    if not chunks:
        skipped += 1
        continue

    emb = encode_chunks(chunks, model, tokenizer, DEVICE, BATCH_SIZE)
    embeddings[idx] = emb
    has_note[idx] = 1.0
    processed += 1

    if processed % 500 == 0:
        elapsed = time.time() - t0
        rate = processed / elapsed
        remaining = (len(note_texts) - processed - skipped) / max(rate, 0.1)
        print(f"  {processed:,} encoded, {skipped:,} skipped | "
              f"{rate:.1f} notes/s | ~{remaining/60:.0f} min remaining")

elapsed = time.time() - t0
print(f"\nDone! {processed:,} notes encoded in {elapsed/60:.1f} min")
print(f"Coverage: {(has_note.sum()/N)*100:.1f}% of admissions have notes")

In [ ]:
# Save embeddings
output = {
    "embeddings": embeddings,   # (N, 768)
    "has_note": has_note,       # (N,)
    "hadm_ids": hadm_ids,
    "method": "clinicalbert_meanpool_normweighted",
    "model_name": MODEL_NAME,
    "chunk_size": MAX_LENGTH,
    "overlap": OVERLAP,
    "num_encoded": int(has_note.sum()),
}

out_path = OUTPUT_DIR / f"note_embeddings_mimic{MIMIC_VERSION}.pkl"
with open(out_path, "wb") as f:
    pickle.dump(output, f)
print(f"Saved to {out_path}")
print(f"  Shape: {embeddings.shape}")
print(f"  Mean L2 norm (encoded notes): {np.linalg.norm(embeddings[has_note==1], axis=1).mean():.2f}")

# Compute note_global_mean from TRAINING SET ONLY for mean-centering.
# This is needed by GatedFusion and MultiHeadCopyPredictor to fix ClinicalBERT anisotropy.
# Load records to get the sequential train split (first 2/3 of patients).
records_path = INPUT_DIR / f"records_final.pkl"
if records_path.exists():
    with open(records_path, "rb") as f:
        records = pickle.load(f)
    n_patients = len(records)
    n_train = int(n_patients * 2 / 3)
    # Collect train hadm_ids
    train_hadm_ids = set()
    for patient in records[:n_train]:
        for visit in patient:
            if len(visit) > 3:
                train_hadm_ids.add(int(visit[3]))
    # Compute mean from training notes only
    train_mask = np.array([int(h) in train_hadm_ids for h in hadm_ids])
    train_encoded = (has_note == 1) & train_mask
    if train_encoded.sum() > 0:
        note_global_mean = embeddings[train_encoded].mean(axis=0)  # (768,)
        mean_path = OUTPUT_DIR / "note_global_mean.npy"
        np.save(mean_path, note_global_mean)
        
        # Verify centering quality
        centered = embeddings[train_encoded] - note_global_mean
        norms = np.linalg.norm(centered, axis=1, keepdims=True)
        norms = np.clip(norms, 1e-8, None)
        normed = centered / norms
        cos_sim = (normed @ normed.T)
        n = cos_sim.shape[0]
        off_diag = cos_sim[np.triu_indices(n, k=1)]
        print(f"\n  note_global_mean saved to {mean_path}")
        print(f"  Training notes used: {train_encoded.sum():,}")
        print(f"  Global mean norm: {np.linalg.norm(note_global_mean):.4f}")
        print(f"  Cos-sim BEFORE centering: (computed at extraction time)")
        print(f"  Cos-sim AFTER centering:  mean={off_diag.mean():.4f}, std={off_diag.std():.4f}")
else:
    print(f"\n  WARNING: {records_path} not found — cannot compute note_global_mean.")
    print(f"  Upload records_final.pkl or run center_note_embeddings.py locally.")


In [ ]:
# Quick sanity check
print("Sample embedding stats:")
encoded_mask = has_note == 1
print(f"  Encoded notes: {encoded_mask.sum():,}")
print(f"  Embedding range: [{embeddings[encoded_mask].min():.4f}, {embeddings[encoded_mask].max():.4f}]")
print(f"  Mean: {embeddings[encoded_mask].mean():.4f}")
print(f"  Std: {embeddings[encoded_mask].std():.4f}")